In [2]:
pip install gspread oauth2client requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.1 MB/s eta 0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 2.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.5/236.5 kB 3.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.4/83.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.3/181.3 kB 2.6 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 5.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.1/160.1 kB 6.8 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# 1. Setup the credentials
# Ensure 'service_account.json' is in the same folder as this notebook!
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]

try:
    creds = ServiceAccountCredentials.from_json_keyfile_name('service_account.json', scope)
    client = gspread.authorize(creds)
    
    # 2. Open the sheet by name 
    # (Make sure you shared the sheet with the email inside the JSON file!)
    sheet_name = "LiveNiftyTrackerSymbol" 
    sheet = client.open(sheet_name).sheet1
    
    # 3. Test Write
    sheet.update_acell('G1', 'Connection Successful!')
    print(f"✅ SUCCESS: Python has successfully written to '{sheet_name}'")
    print("Check your Google Sheet cell G1 now!")

except FileNotFoundError:
    print("❌ ERROR: 'service_account.json' file not found in this folder.")
except gspread.exceptions.SpreadsheetNotFound:
    print(f"❌ ERROR: Could not find a sheet named '{sheet_name}'. Check the spelling.")
except Exception as e:
    print(f"❌ ERROR: {e}")

✅ SUCCESS: Python has successfully written to 'LiveNiftyTrackerSymbol'
Check your Google Sheet cell G1 now!


In [5]:
import upstox_client

# REPLACE THESE with your details from Upstox Developer Console
api_key = '76ef74f9-6187-449d-9b2f-9909e0950423'
redirect_uri = 'https://127.0.0.1:5000/' # Or whatever you set in the dashboard

print("1. Click this link and log in:")
print(f"https://api.upstox.com/v2/login/authorization/dialog?client_id={api_key}&redirect_uri={redirect_uri}&response_type=code")

1. Click this link and log in:
https://api.upstox.com/v2/login/authorization/dialog?client_id=76ef74f9-6187-449d-9b2f-9909e0950423&redirect_uri=https://127.0.0.1:5000/&response_type=code


In [6]:
import requests

auth_code = 'IV-qpL'
api_secret = 'oytwczd85q'

url = 'https://api.upstox.com/v2/login/authorization/token'
data = {
    'code': auth_code,
    'client_id': api_key,
    'client_secret': api_secret,
    'redirect_uri': redirect_uri,
    'grant_type': 'authorization_code',
}

response = requests.post(url, data=data)
res_json = response.json()

if 'access_token' in res_json:
    ACCESS_TOKEN = res_json['access_token']
    print("Success! Your Access Token is ready.")
else:
    print("Error:", res_json)

Success! Your Access Token is ready.


In [13]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import requests
import time
from datetime import datetime

# --- 1. CONFIGURATION ---
symbols = ["RELIANCE", "TCS", "HDFCBANK", "ICICIBANK", "INFY"]
# We generate both possible key formats to ensure a match
instrument_keys = [f"NSE_EQ|{s}" for s in symbols]
data_store = {s: {'high': 0, 'low': 999999, 'ltp': 0, 'time': ""} for s in symbols}

# --- 2. CONNECT TO GOOGLE SHEET ---
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
try:
    creds = ServiceAccountCredentials.from_json_keyfile_name('service_account.json', scope)
    client = gspread.authorize(creds)
    # MAC FIX: Force opening by the exact title seen in your browser
    workbook = client.open("LiveNiftyTrackerSymbol")
    # Finding worksheet by title to avoid index confusion
    sheet = workbook.worksheet("Sheet1") 
    print(f"✅ Successfully linked to: {workbook.title} > Sheet1")
except Exception as e:
    print(f"❌ Connection Error: {e}")
    exit()

# --- 3. THE IMPROVED DATA ENGINE ---
def fetch_and_force_push():
    keys_str = ",".join(instrument_keys)
    url = f'https://api.upstox.com/v2/market-quote/quotes?instrument_key={keys_str}'
    headers = {'Authorization': f'Bearer {ACCESS_TOKEN}', 'Accept': 'application/json'}
    
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            api_data = response.json().get('data', {})
            now = datetime.now()
            current_candle = now.replace(minute=(now.minute // 5) * 5, second=0, microsecond=0).strftime('%H:%M')
            
            rows = []
            for raw_key, details in api_data.items():
                # SAFE PARSING: Removes both ":" and "|" to get the symbol name
                clean_sym = raw_key.replace("NSE_EQ:", "").replace("NSE_EQ|", "")
                
                if clean_sym in data_store:
                    ltp = details['last_price']
                    
                    # 5-min Candle Logic
                    if data_store[clean_sym]['time'] != current_candle:
                        data_store[clean_sym] = {'high': ltp, 'low': ltp, 'ltp': ltp, 'time': current_candle}
                    else:
                        data_store[clean_sym]['high'] = max(data_store[clean_sym]['high'], ltp)
                        data_store[clean_sym]['low'] = min(data_store[clean_sym]['low'], ltp)
                        data_store[clean_sym]['ltp'] = ltp
                    
                    rows.append([
                        clean_sym, ltp, data_store[clean_sym]['high'], 
                        data_store[clean_sym]['low'], current_candle, 
                        now.strftime('%H:%M:%S')
                    ])
            
            if rows:
                # v6.0 syntax: named parameters to avoid index/order errors
                sheet.update(range_name='A2:F6', values=rows)
                print(f"✅ Sheet Synced: {now.strftime('%H:%M:%S')} | Reliance: {rows[0][1]}")
            
        else:
            print(f"⚠️ API Error {response.status_code}")
            
    except Exception as e:
        print(f"❌ Python Error: {e}")

# --- 4. START ---
# Write headers once
sheet.update(range_name='A1', values=[["Symbol", "LTP", "5m High", "5m Low", "Candle", "Last Sync"]])

print("🚀 Live Stream Started. Check your Google Sheet!")
try:
    while True:
        fetch_and_force_push()
        time.sleep(5) # Refresh every 5s for Google Sheets safety
except KeyboardInterrupt:
    print("🛑 Stopped.")

✅ Successfully linked to: LiveNiftyTrackerSymbol > Sheet1
🚀 Live Stream Started. Check your Google Sheet!
🛑 Stopped.


In [10]:
# --- 1. FORCE-OPEN BY NAME ---
# Instead of .sheet1, we find the tab named 'Sheet1' (case sensitive)
try:
    workbook = client.open("LiveNiftyTrackerSymbol")
    sheet = workbook.worksheet("Sheet1") 
    print("✅ Found 'Sheet1' successfully.")
except Exception as e:
    # If it can't find 'Sheet1', it will list all available tabs
    print(f"❌ Error finding 'Sheet1'. Available tabs: {[s.title for s in workbook.worksheets()]}")
    sheet = workbook.get_worksheet(0) # Fallback to first tab

# --- 2. THE FORCE-PUSH FUNCTION ---
def force_push():
    # ... (Keep your existing Upstox fetching logic here) ...
    
    if response.status_code == 200:
        # ... (Keep your data processing logic here) ...

        # CHANGE: We will clear the range first to force the UI to refresh
        # Then we write to a specific named range
        cell_range = f'A2:F{len(rows) + 1}'
        sheet.batch_clear([cell_range]) # Clears just the data area
        sheet.update(range_name=cell_range, values=rows)
        
        print(f"✅ Data Sent to Sheet: {now.strftime('%H:%M:%S')}")

✅ Found 'Sheet1' successfully.


In [14]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials

scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]

try:
    creds = ServiceAccountCredentials.from_json_keyfile_name('service_account.json', scope)
    client = gspread.authorize(creds)
    
    # 1. Identify the Workbook
    workbook = client.open("LiveNiftyTrackerSymbol")
    print(f"--- DIAGNOSTICS FOR: {workbook.title} ---")
    print(f"Workbook ID: {workbook.id}") # Compare this to the ID in your browser URL
    
    # 2. List all visible tabs (Worksheets)
    worksheets = workbook.worksheets()
    print(f"Number of tabs found: {len(worksheets)}")
    for i, s in enumerate(worksheets):
        print(f"Tab {i}: Title='{s.title}', ID={s.id}")

    # 3. Test Write to the VERY FIRST TAB
    target_sheet = worksheets[0]
    print(f"\nAttempting test write to tab: '{target_sheet.title}'...")
    target_sheet.update_acell('Z1', 'DEBUG_TEST_SUCCESS')
    
    # 4. Read back what is in A1
    a1_val = target_sheet.acell('A1').value
    print(f"The current value in cell A1 of this tab is: '{a1_val}'")

except Exception as e:
    print(f"❌ DIAGNOSTIC FAILED: {e}")

--- DIAGNOSTICS FOR: LiveNiftyTrackerSymbol ---
Workbook ID: 1EW-AQv4cMK9atC9y1hcVHzeTn3omUlQ1XXPhuaG1rAk
Number of tabs found: 1
Tab 0: Title='Sheet1', ID=0

Attempting test write to tab: 'Sheet1'...
The current value in cell A1 of this tab is: 'Symbol'


In [25]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import requests
import time
from datetime import datetime

# --- 1. CONFIG ---
target_isin = "NSE_EQ|INE002A01018"
friendly_name = "RELIANCE"
data_store = {friendly_name: {'high': 0, 'low': 999999, 'ltp': 0, 'time': ""}}

# --- 2. SECURE CONNECTION ---
def get_sheet():
    scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
    try:
        creds = ServiceAccountCredentials.from_json_keyfile_name('service_account.json', scope)
        client = gspread.authorize(creds)
        # Directly using the ID you verified
        return client.open_by_key("1EW-AQv4cMK9atC9y1hcVHzeTn3omUlQ1XXPhuaG1rAk").worksheet("Sheet2")
    except Exception as e:
        print(f"❌ Connection Setup Error: {e}")
        return None

sheet = get_sheet()

# --- 3. THE RESILIENT DATA ENGINE ---
def run_reliance_tracker():
    url = f'https://api.upstox.com/v2/market-quote/quotes?instrument_key={target_isin}'
    headers = {'Authorization': f'Bearer {ACCESS_TOKEN}', 'Accept': 'application/json'}
    
    try:
        # Added a 10-second timeout to prevent the script from hanging
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code == 200:
            api_data = response.json().get('data', {})
            now = datetime.now()
            candle = now.replace(minute=(now.minute // 5) * 5, second=0, microsecond=0).strftime('%H:%M')
            
            # Handle both pipe and colon formats
            found_key = next((k for k in [target_isin, target_isin.replace('|', ':')] if k in api_data), None)

            if found_key:
                price = api_data[found_key]['last_price']
                
                # Update High/Low
                if data_store[friendly_name]['time'] != candle:
                    data_store[friendly_name] = {'high': price, 'low': price, 'ltp': price, 'time': candle}
                else:
                    data_store[friendly_name]['high'] = max(data_store[friendly_name]['high'], price)
                    data_store[friendly_name]['low'] = min(data_store[friendly_name]['low'], price)
                    data_store[friendly_name]['ltp'] = price
                
                row_data = [friendly_name, price, data_store[friendly_name]['high'], 
                            data_store[friendly_name]['low'], candle, now.strftime('%H:%M:%S')]

                # Pushing to Sheet with a retry for SSLErrors
                try:
                    sheet.update(values=[row_data], range_name='A2:F2', value_input_option='USER_ENTERED')
                    sheet.update_acell('J1', f"Last Sync: {now.strftime('%H:%M:%S')}")
                    print(f"✅ RELIANCE: {price} | Sync: {now.strftime('%H:%M:%S')}")
                except Exception as sheet_err:
                    print(f"⚠️ Sheet write failed (SSL/Internet), retrying in next loop...")
            
    except requests.exceptions.SSLError:
        print("⚠️ SSL Error detected. Refreshing connection...")
        time.sleep(2)
    except Exception as e:
        print(f"❌ Loop Error: {e}")

# --- 4. START ---
if sheet:
    sheet.clear()
    sheet.update(values=[["Symbol", "LTP", "5m High", "5m Low", "Candle", "Sync Time"]], 
                 range_name='A1', value_input_option='USER_ENTERED')
    
    print(f"🚀 Monitoring Reliance. Check Google Sheets (Sheet2)!")
    try:
        while True:
            run_reliance_tracker()
            time.sleep(5) 
    except KeyboardInterrupt:
        print("Stopped.")

❌ Connection Setup Error: HTTPSConnectionPool(host='sheets.googleapis.com', port=443): Max retries exceeded with url: /v4/spreadsheets/1EW-AQv4cMK9atC9y1hcVHzeTn3omUlQ1XXPhuaG1rAk?includeGridData=false (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1006)')))


In [26]:
import os
import sys
import subprocess

# This identifies the exact Python version your VS Code is using
python_exe = sys.executable
install_cert_path = python_exe.replace("bin/python", "bin/python -m certifi")

print(f"🔍 Current Python: {python_exe}")

try:
    # 1. Update the 'certifi' package first
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "certifi"])
    
    # 2. Force the kernel to use the updated certificates
    import certifi
    os.environ['SSL_CERT_FILE'] = certifi.where()
    
    print(f"✅ SSL Certificates updated and linked to: {certifi.where()}")
except Exception as e:
    print(f"❌ Certificate update failed: {e}")

🔍 Current Python: /usr/local/bin/python3
✅ SSL Certificates updated and linked to: /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/certifi/cacert.pem



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [32]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import requests
import certifi
import time
import os
from datetime import datetime

# --- 1. SSL FIX ---
os.environ['SSL_CERT_FILE'] = certifi.where()

# --- 2. CONFIG ---
isin_to_find = "INE002A01018"
target_key = f"NSE_EQ|{isin_to_find}"
sheet_id = "1EW-AQv4cMK9atC9y1hcVHzeTn3omUlQ1XXPhuaG1rAk"
data_store = {"RELIANCE": {'high': 0, 'low': 999999, 'ltp': 0, 'time': ""}}

# --- 3. THE FLEXIBLE PUSH ENGINE ---
def run_flexible_push():
    url = f"https://api.upstox.com/v2/market-quote/quotes?instrument_key={target_key}"
    headers = {'Authorization': f'Bearer {ACCESS_TOKEN}', 'Accept': 'application/json'}
    
    try:
        # Fetch data from Upstox
        response = requests.get(url, headers=headers, timeout=5)
        
        if response.status_code == 200:
            api_payload = response.json().get('data', {})
            
            # DYNAMIC KEY PICKER: Find the key that contains our ISIN
            # This handles cases where Upstox returns ':' instead of '|'
            actual_key = next((k for k in api_payload.keys() if isin_to_find in k), None)
            
            if actual_key:
                price = api_payload[actual_key]['last_price']
                now = datetime.now()
                candle = now.replace(minute=(now.minute // 5) * 5, second=0).strftime('%H:%M')
                
                # Update High/Low
                d = data_store["RELIANCE"]
                if d['time'] != candle:
                    data_store["RELIANCE"] = {'high': price, 'low': price, 'ltp': price, 'time': candle}
                else:
                    d['high'], d['low'], d['ltp'] = max(d['high'], price), min(d['low'], price), price

                # Connect and Push to Google Sheets
                scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
                creds = ServiceAccountCredentials.from_json_keyfile_name('service_account.json', scope)
                client = gspread.authorize(creds)
                sheet = client.open_by_key(sheet_id).worksheet("Sheet2")
                
                row = ["RELIANCE", price, data_store["RELIANCE"]['high'], data_store["RELIANCE"]['low'], candle, now.strftime('%H:%M:%S')]
                sheet.update(values=[row], range_name='A2:F2', value_input_option='USER_ENTERED')
                sheet.update_acell('J1', f"Last Sync: {now.strftime('%H:%M:%S')}")
                
                print(f"✅ RELIANCE ({actual_key}): {price} | Updated at {now.strftime('%H:%M:%S')}")
            else:
                print(f"⚠️ ISIN {isin_to_find} not found. Keys in response: {list(api_payload.keys())}")
        else:
            print(f"⚠️ Upstox Error {response.status_code}: {response.text}")

    except Exception as e:
        print(f"❌ Error: {e}")

# --- 4. START ---
print("🚀 Starting Stream. Monitoring Reliance...")
try:
    while True:
        run_flexible_push()
        time.sleep(5) 
except KeyboardInterrupt:
    print("Stopped.")

🚀 Starting Stream. Monitoring Reliance...
⚠️ ISIN INE002A01018 not found. Keys in response: ['NSE_EQ:RELIANCE']
⚠️ ISIN INE002A01018 not found. Keys in response: ['NSE_EQ:RELIANCE']
Stopped.


In [29]:
import pandas as pd
import requests
import gzip
import io

# Official Upstox Master Instrument URL
url = "https://assets.upstox.com/market-quote/instruments/exchange/NSE.json.gz"

print("📥 Downloading latest instrument master...")
response = requests.get(url)
response.raise_for_status()

# Decompress and load into a DataFrame
with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as f:
    data = f.read().decode("utf-8")
    df = pd.read_json(io.StringIO(data))

# Search for RELIANCE specifically in Equity
reliance_data = df[(df['trading_symbol'] == 'RELIANCE') & (df['segment'] == 'NSE_EQ')]

if not reliance_data.empty:
    exact_key = reliance_data.iloc[0]['instrument_key']
    print(f"✅ FOUND! Use this exact key: {exact_key}")
else:
    print("❌ Could not find RELIANCE in the NSE_EQ segment.")

📥 Downloading latest instrument master...
✅ FOUND! Use this exact key: NSE_EQ|INE002A01018
